# MR604 — Four-Metric SNN Benchmark

**Deepak Gajmer (MIT251589) · Melbourne Institute of Technology**

Produces every value Chapter 5 currently marks `[MEASURED VALUE REQUIRED]`.

**Before anything: Runtime → Change runtime type → GPU.** Cell 1 refuses to continue
on CPU. Run cells in order with **Shift+Enter**.


## 1 · GPU check


In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), (
    'No GPU. Runtime -> Change runtime type -> Hardware accelerator: GPU, then rerun.')
print('GPU  :', torch.cuda.get_device_name(0))
print('VRAM :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
print('torch:', torch.__version__)


## 2 · Mount Drive

The machine is temporary and its local disk is wiped on disconnect. Checkpoints go
to Drive so a dropped session costs one epoch, not the whole run.


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os
BASE    = '/content/drive/MyDrive/MR604'
CODE    = f'{BASE}/code'
RESULTS = f'{BASE}/results'
CKPTS   = f'{BASE}/checkpoints'
TABLES  = f'{BASE}/tables'
FIGURES = f'{BASE}/figures'
DATA    = '/content/data'      # dataset is 11 MB; keep it local, it is faster
for d in (RESULTS, CKPTS, TABLES, FIGURES, DATA): os.makedirs(d, exist_ok=True)
assert os.path.exists(f'{CODE}/run.py'), f'run.py not in {CODE} -- upload code/ to Drive'
os.chdir(CODE); import sys; sys.path.insert(0, CODE)
print('cwd:', os.getcwd())


## 3 · Install


In [ ]:
!pip install -q spikingjelly==0.0.0.0.14 fvcore tqdm tabulate

# NOTE: spikingjelly.__version__ does NOT exist -- the top-level package is empty,
# so reaching for it raises AttributeError. Use importlib.metadata, which reads the
# installed distribution metadata and works for any pip-installed package.
import importlib.metadata as md
for pkg in ('torch', 'torchvision', 'spikingjelly', 'fvcore'):
    try:
        print(f'{pkg:<14}', md.version(pkg))
    except md.PackageNotFoundError:
        print(f'{pkg:<14} NOT INSTALLED')


In [ ]:
# Full environment check: versions, GPU, and the specific SpikingJelly symbols this
# code calls. A version string alone proves little -- this confirms the API exists
# and that ParametricLIFNode accepts v_reset (needed for soft reset).
!python tools/check_install.py


## 4 · Get the dataset

**You do not download MNIST manually.** `torchvision` fetches it automatically on
first use — about 11 MB, a few seconds on Colab. It tries
`ossci-datasets.s3.amazonaws.com` first and falls back to `yann.lecun.com`.

This cell does it explicitly and verifies the result, so a download problem shows up
here in 10 seconds rather than 20 minutes into training. It must print
`dataset check: PASS`.


In [ ]:
!python run.py data --data-root {DATA}


### If the download fails

Rare, but if both mirrors are blocked, fetch the four files manually and drop them in
`{DATA}/MNIST/raw/`. Do not unzip them — torchvision expects the `.gz` files.


In [ ]:
# Manual fallback -- only run this if the cell above FAILED.
!mkdir -p {DATA}/MNIST/raw && cd {DATA}/MNIST/raw && \
 wget -q https://ossci-datasets.s3.amazonaws.com/mnist/train-images-idx3-ubyte.gz && \
 wget -q https://ossci-datasets.s3.amazonaws.com/mnist/train-labels-idx1-ubyte.gz && \
 wget -q https://ossci-datasets.s3.amazonaws.com/mnist/t10k-images-idx3-ubyte.gz  && \
 wget -q https://ossci-datasets.s3.amazonaws.com/mnist/t10k-labels-idx1-ubyte.gz  && \
 md5sum *.gz

# Expected MD5s:
#   f68b3c2dcbeaaa9fbdd348bbdeb94873  train-images-idx3-ubyte.gz
#   d53e105ee54ea40749a09fcbcd1e9432  train-labels-idx1-ubyte.gz
#   9fb629c4189551a2d022fa330f9573f3  t10k-images-idx3-ubyte.gz
#   ec29112dd5afa0611ce80d1b7f02629c  t10k-labels-idx1-ubyte.gz


## 5 · Verify the operation counts

Trains nothing; takes seconds. **Must print `verification: PASS`.**

Every energy number in the thesis is downstream of these counts. This is the check
that would have caught the candidature-stage error (a pJ→nJ slip plus a 4.24× MAC
undercount).

fvcore will report 4,316,672 against the analytical 4,241,152. The 75,520 gap is
BatchNorm affine ops, which the SNN convention folds into the preceding convolution.
The script predicts that number exactly rather than allowing a tolerance.


In [ ]:
!python run.py verify


In [ ]:
# Confirms the native neuron implementations match SpikingJelly's exactly.
# Run once and keep the output -- it is your answer if a reviewer asks whether the
# results are an artefact of the framework. If it FAILS, add --backend native
# to every command below; that path is fully tested.
!python tools/check_backends.py


## 6 · Smoke test — ~20 min

Every configuration, 2 epochs, one seed. This is the *test run*: it proves the
pipeline, monitors, gates and logging work before you commit 7 hours.

Accuracy will be below the final figures (2 epochs is not 64) but should look sane —
roughly 97–99% for the ANN, spiking configs firing normally.


In [ ]:
!python run.py smoke \
    --out-dir {BASE}/results_smoke \
    --ckpt-dir {BASE}/ckpt_smoke \
    --data-root {DATA}


In [ ]:
# Automated plumbing check.
import json, glob
ok = True
for f in sorted(glob.glob(f'{BASE}/results_smoke/*.json')):
    if f.split('/')[-1].startswith('_'): continue
    r = json.load(open(f))
    spiking = r['pathway'] != 'ann'
    ratio = r['e_si_uj']/r['e_th_uj'] if r['e_th_uj'] else 0
    checks = {
        'layer_rates'    : (r.get('spike_rate_per_layer') is not None) == spiking,
        'sop_positive'   : (r.get('n_sop', 0) > 0) == spiking,
        'energy_ratio_2.10': (abs(ratio-2.10) < 1e-6) if spiking else True,
        'three_latencies': all(r.get(k) is not None for k in ['latency_gpu_ms','latency_cpu_ms']
                               + (['latency_neuromorphic_projected_ms'] if spiking else [])),
        'accuracy_sane'  : r['accuracy'] > 20.0,
        'no_warnings'    : not r.get('warnings'),
    }
    bad = [k for k,v in checks.items() if not v]
    ok &= not bad
    print(f"{r['tag']:<22}{r['accuracy']:>7.2f}%  {'OK' if not bad else 'FAIL: '+','.join(bad)}")
print()
print('SMOKE TEST:', 'PASS -- proceed' if ok else 'FAIL -- fix before the full grid')


### If you see `ZERO SPIKES` or ~10% accuracy

Stop. On full MNIST this should never happen, and it is not something to wave
through.

The cause is almost always unconverged BatchNorm running statistics. The running
variance starts at 1.0 and decays as `0.9^k`; at low `k` the eval-mode activations
are divided by a variance far too large, the network predicts a constant class, and
*training-mode* accuracy still looks healthy — so it is invisible unless you check.
Measured on this code with the plain ANN:

| BatchNorm updates | eval acc | train-mode acc |
|---|---|---|
| 16 | 11.7% | 89.7% |
| 51 | 57.1% | 95.6% |
| 96 | **98.3%** | 98.1% |

Full MNIST at batch 256 gives **235 updates per epoch**, so one epoch clears it and
the 2-epoch smoke test (470 updates) is well past it. This affects every pathway, not
just Path A. If you see it on the real dataset, you are training on a subset, using a
very large batch, or something is broken.


In [ ]:
# Clear the smoke results so they cannot contaminate the real ones.
!rm -rf {BASE}/results_smoke {BASE}/ckpt_smoke
print('cleared')


## 7 · Full grid — ~7 h on a T4

21 runs: 3 ANN + 9 Path A + 9 Path B. Path B is additionally evaluated under both
input encodings at no training cost, giving 30 result rows.

**Re-run this same cell after any disconnect.** Finished runs are skipped; partial
runs resume from the last epoch checkpoint.

| Configuration | per seed | ×3 seeds |
|---|---|---|
| ANN baseline | ~4 min | 12 min |
| QCFS source ANN (T=2,4,8) | ~5 min each | 45 min |
| STBP T=4 | ~18 min | 55 min |
| STBP T=8 | ~35 min | 1 h 45 |
| STBP T=16 | ~70 min | 3 h 30 |

Colab's idle timeout watches *browser activity*, not your job. Keep the tab open and
disable PC sleep, or it will be killed mid-run.


In [ ]:
!python run.py grid --epochs 64 \
    --out-dir {RESULTS} --ckpt-dir {CKPTS} -r-data-root {DATA}


In [ ]:
# Short on time? Use this instead: ANN + STBP T=8 + QCFS T=4, all 3 seeds, ~2.5 h.
# !python run.py priority --epochs 64 \
#     --out-dir {RESULTS} --ckpt-dir {CKPTS} --data-root {DATA}


## 8 · Re-test a trained model without retraining

`eval` loads a checkpoint and re-runs the full four-metric evaluation. Use it to
inspect one configuration, or to re-measure after changing something in the
evaluation code. It warns if the checkpoint came from an incomplete run.


In [ ]:
!python run.py eval --pathway stbp --T 8 --seed 42 \
    --ckpt-dir {CKPTS} --data-root {DATA}


## 9 · Gates, tables, figures

`tables/thesis_tables.md` is paste-ready for Chapter 5.


In [ ]:
!python run.py gates --out-dir {RESULTS}


In [ ]:
!python analyse.py --out-dir {RESULTS} --tables-dir {TABLES} --figures-dir {FIGURES}


In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open(f'{TABLES}/thesis_tables.md').read()))
for f in ('fig_training_curves','fig_accuracy_energy','fig_spike_rates'):
    display(Image(f'{FIGURES}/{f}.png'))


## What to expect, and what not to flinch at

**Path B is reported under two input encodings and you must report both.** Measured
on this code with a trained source network:

| Input encoding | Conversion loss | Why |
|---|---|---|
| rate (matched with Path A) | large — 42 pp at T=4 | Bernoulli sampling at small T is noisy |
| direct (what Bu et al. use) | 0 pp at T=4 and T=8 | no sampling noise |

Direct encoding means conv1 receives analogue values and runs **MACs at every
timestep**, so energy roughly doubles and the closed-form break-even test stops being
sufficient — G1 falls back to a direct energy comparison, which the code handles and
flags. Neither encoding is *the* answer. This trade-off is a genuine finding the
thesis does not yet discuss.

**At T=16 the calibrated break-even threshold is 0.152.** Your candidature pilot
measured a *network mean* of 0.121 — passing by only 21%. G1 here is applied **per
layer**, and conv2 carries 85.2% of all operations. If conv2 fires above 0.152, STBP
T=16 fails at the calibrated constant regardless of the mean.

**A 4-bit quantised ANN may beat every SNN configuration at 1.89 pJ/SOP.**
`analyse.py` prints a note if so. That is a valid answer to RQ1, not a failure —
Chapter 6.2 already states the decision rule leading to it.

**Four things in this code deliberately differ from the thesis draft** — exact SOP
counting, the QCFS shift term, the `SOPMonitor` naming, and quantised-baseline energy
scaling. See *Four things* in `README.md`. Update the thesis to match; do not leave
them inconsistent.
